In [1]:

rf_mae = 1406.8289158625016
rf_rmse = 1797.7297012268043
rf_r2 = 0.9982142133095491 


In [2]:
lr_mae = 592.3732484747742
lr_rmse = 742.5901496523799
lr_r2 = 0.9996952952933795

In [ ]:
gb_mae = 884.8472970961152
gb_rmse = 1136.1994282243343
gb_r2 = 0.9992866709230596

In [6]:
import pandas as pd


In [10]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

df = pd.read_csv("kenya_motor_insurance_dataset.csv")

X = df.drop(
    columns=["policy_id", "premium_amount"]
)

y = df["premium_amount"]

categorical_cols = X.select_dtypes(
    include=["object"]
).columns

numeric_cols = X.select_dtypes(
    exclude=["object"]
).columns

preprocessor = ColumnTransformer([
    (
        "cat",
        OneHotEncoder(handle_unknown="ignore"),
        categorical_cols
    ),
    (
        "num",
        "passthrough",
        numeric_cols
    )
])

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Data preparation complete.")

Data preparation complete.


In [11]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

gb_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", GradientBoostingRegressor(
        n_estimators=200,
        learning_rate=0.05,
        random_state=42
    ))
])

gb_pipeline.fit(X_train, y_train)

gb_predictions = gb_pipeline.predict(X_test)

gb_mae = mean_absolute_error(y_test, gb_predictions)
gb_rmse = mean_squared_error(y_test, gb_predictions) ** 0.5
gb_r2 = r2_score(y_test, gb_predictions)

print("Gradient Boosting")
print("MAE:", gb_mae)
print("RMSE:", gb_rmse)
print("R2:", gb_r2)

Gradient Boosting
MAE: 884.8472970961152
RMSE: 1136.1994282243343
R2: 0.9992866709230596


In [12]:
results = {
    "Model": [
        "Linear Regression",
        "Random Forest",
        "Gradient Boosting"
    ],
    "MAE": [
        lr_mae,
        rf_mae,
        gb_mae
    ],
    "RMSE": [
        lr_rmse,
        rf_rmse,
        gb_rmse
    ],
    "R2 Score": [
        lr_r2,
        rf_r2,
        gb_r2
    ]
}

comparison = pd.DataFrame(results)

print(comparison)

comparison.to_csv(
    "model_comparison.csv",
    index=False
)

               Model          MAE         RMSE  R2 Score
0  Linear Regression   592.373248   742.590150  0.999695
1      Random Forest  1406.828916  1797.729701  0.998214
2  Gradient Boosting   884.847297  1136.199428  0.999287
